# Processo de Decisão de Markov (MDP) — Grid World

Um **Processo de Decisão de Markov (MDP)** modela problemas de decisão sequencial sob incerteza. Ao contrário da busca determinística, as ações têm resultados estocásticos — o agente não sabe exatamente para qual estado irá ao executar uma ação.

## Definição Formal

Um MDP é uma 5-tupla **(S, A, T, R, γ)**:

| Componente | Descrição |
|---|---|
| **S** | Conjunto de estados |
| **A** | Conjunto de ações |
| **T(s, a, s')** | Probabilidade de transição: P(s' \| s, a) |
| **R(s, a, s')** | Recompensa imediata ao transitar de s para s' via a |
| **γ ∈ (0,1]** | Fator de desconto (quanto o agente valoriza recompensas futuras) |

## Objetivo

Encontrar uma **política ótima** π*: S → A que maximize o **retorno esperado descontado**:

**V\*(s) = max_a Σ_{s'} T(s,a,s') [R(s,a,s') + γ · V\*(s')]**

Esta é a **Equação de Bellman** — base dos algoritmos de programação dinâmica para MDPs.

## Grid World

O ambiente é uma grade 3×4 com:
- **Paredes** — células inacessíveis
- **Terminais** — estados absorventes com recompensa fixa
- **Ações estocásticas** — a ação pretendida ocorre com probabilidade `p_intended`; o agente pode "escorregar" para as direções perpendiculares com probabilidade `(1-p_intended)/3`

## Algoritmos

| Algoritmo | Descrição |
|---|---|
| **Iteração de Valor** | Resolve as equações de Bellman iterativamente até convergência |
| **Política Gulosa** | Extrai π* a partir de V* escolhendo a melhor ação em cada estado |

In [1]:
from structures.problems.grid_world import GridWorld
from structures.algorithms.mdp_solver import value_iteration, greedy_policy_from_v

## Configuração do Grid World

O ambiente é um grid 3×4 com as seguintes características:

```
     col 0   col 1   col 2   col 3
row 0  [ ]     [ ]     [ ]    [+1] ← terminal positivo
row 1  [ ]    [###]    [ ]    [-1] ← terminal negativo
row 2  [ ]     [ ]     [ ]    [ ]
```

| Parâmetro | Valor | Descrição |
|---|---|---|
| `walls` | {(1,1)} | Célula bloqueada (parede central) |
| `terminals` | {(0,3), (1,3)} | Estados absorventes |
| `terminal_rewards` | (0,3)=+1.0, (1,3)=-1.0 | Recompensa ao entrar no terminal |
| `step_reward` | 1.4 | Recompensa por cada passo não-terminal |
| `p_intended` | 0.8 | Probabilidade de mover na direção pretendida |
| `gamma` | 0.99 | Fator de desconto |

**Nota sobre `step_reward=1.4`:** ao contrário do grid world clássico (onde `step_reward=-0.04`), aqui o agente é **recompensado positivamente** por cada passo. Isso cria uma política contraintuitiva: o agente pode preferir permanecer em movimento a chegar rapidamente ao terminal positivo.

In [10]:
mdp = GridWorld(
    n_rows=3,
    n_cols=4,
    walls={(1, 1)},
    terminals={(0, 3), (1, 3)},
    terminal_rewards={(0, 3): 1.0, (1, 3): -1.0},
    step_reward=1.4,
    p_intended=0.8,
    gamma=0.99,
    seed=42
)

## Iteração de Valor (*Value Iteration*)

### Algoritmo de Bellman

A Iteração de Valor atualiza repetidamente os valores de todos os estados até convergência:

**V_{k+1}(s) = max_a Σ_{s'} T(s,a,s') [R(s,a,s') + γ · V_k(s')]**

O algoritmo converge para V* quando a variação máxima entre iterações é menor que θ:

```
δ = max_s |V_{k+1}(s) - V_k(s)| < θ
```

### Parâmetro `theta=1e-10`

Com `θ=1e-10`, a convergência é muito precisa — adequado para análise detalhada da política.

### Interpretação dos Valores

Os valores V*(s) representam o **retorno esperado descontado total** a partir de cada estado seguindo a política ótima:

- V*(0,3) = 1.0 (terminal positivo — valor fixo)
- V*(1,3) = -1.0 (terminal negativo — valor fixo)
- Estados não-terminais terão valores altos (≈139) devido ao `step_reward=1.4` positivo — o agente acumula muita recompensa antes de terminar

In [11]:
V = value_iteration(mdp, theta=1e-10)
V

{(0, 0): 139.4661055085353,
 (0, 1): 138.59877775750113,
 (0, 2): 127.97853834499654,
 (0, 3): 1.0,
 (1, 0): 139.5316417291814,
 (1, 2): 127.09681056006382,
 (1, 3): -1.0,
 (2, 0): 139.52621457684828,
 (2, 1): 139.38930307896968,
 (2, 2): 137.65383526819522,
 (2, 3): 127.02976674239538}

## Política Gulosa (*Greedy Policy*)

A política ótima é extraída de V* pela função `greedy_policy_from_v`:

**π\*(s) = argmax_a Σ_{s'} T(s,a,s') [R(s,a,s') + γ · V\*(s')]**

Para cada estado, a ação escolhida maximiza o **Q-valor**:

**Q\*(s,a) = Σ_{s'} T(s,a,s') [R(s,a,s') + γ · V\*(s')]**

### Análise da Política Resultante

A política exibe comportamento contraintuitivo causado pelo `step_reward=1.4` positivo:

- Estados próximos ao terminal positivo (0,3) se **afastam** (DOWN, LEFT)
- O agente prefere permanecer em estados não-terminais coletando recompensas de passo
- A soma das recompensas descontadas por continuar em movimento supera a recompensa terminal de 1.0

Isso demonstra que a **modelagem das recompensas** afeta dramaticamente o comportamento emergente do agente.

In [12]:
pi = greedy_policy_from_v(mdp, V)
pi

{(0, 0): 'DOWN',
 (0, 1): 'LEFT',
 (0, 2): 'LEFT',
 (0, 3): 'UP',
 (1, 0): 'LEFT',
 (1, 2): 'DOWN',
 (1, 3): 'UP',
 (2, 0): 'UP',
 (2, 1): 'LEFT',
 (2, 2): 'LEFT',
 (2, 3): 'LEFT'}

## Simulação da Política

O agente segue a política π* a partir do estado inicial (0,0) por até 50 passos. A cada passo:

1. Executa a ação indicada por π*(estado_atual)
2. A transição é **estocástica**: `mdp.sample_next(s, a)` sorteia o próximo estado de acordo com T(s,a,·)
3. Acumula a recompensa descontada: G = Σ_t γ^t · r_t

### Comportamento Observado

A simulação mostra que o agente **circula** entre os estados (0,0) e (1,0) sem atingir o terminal em 50 passos. Isso ocorre porque:

1. **`step_reward=1.4` positivo:** cada passo gera recompensa, então a política ótima inclui "ficar em movimento"
2. **Estocasticidade:** com `p_intended=0.8`, o agente frequentemente escorrega para direções não pretendidas
3. **Política DOWN/LEFT:** a política em (0,0)=DOWN e (1,0)=LEFT cria um ciclo natural

O retorno acumulado G ≈ 55 reflete as múltiplas recompensas de passo coletadas. Esta configuração é um exemplo didático importante sobre como o **design de recompensas** (reward shaping) influencia dramaticamente o comportamento de um agente de IA.

In [13]:
s = (0, 0)
G = 0.0
disc = 1.0
path = [s]
for _ in range(50):
    if s in mdp.terminals:
        break

    a = pi[s]
    s2, r, done = mdp.sample_next(s, a)
    print(f"Estado: {s}, Ação: {a}, Próximo estado: {s2}, Recompensa: {r}, Terminal: {done}")
    G += disc * r
    disc *= mdp.gamma
    s = s2
    path.append(s)
    if done:
        break

print(path)
print(G)

Estado: (0, 0), Ação: DOWN, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (0, 0), Recompensa: 1.4, Terminal: False
Estado: (0, 0), Ação: DOWN, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (2, 0), Recompensa: 1.4, Terminal: False
Estado: (2, 0), Ação: UP, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (0, 0), Recompensa: 1.4, Terminal: False
Estado: (0, 0), Ação: DOWN, Próximo estado: (1, 0), Recompensa: 1.4, Terminal: False
Estado: (1, 0), Ação: LEFT, Próximo estado: (1, 0), Recompensa: 1.4